# Notebook 04 - Insights & Interpretability

Covers:
- Era drift analysis (does early data help or hurt?)
- SHAP feature importance (XGBoost)
- Partial dependence plots for top 3 features
- Feature group ablation study
- Home advantage trend (season-by-season with COVID annotations)
- B2B penalty analysis
- is_no_fans_season model effect
- Error analysis (large |predicted_prob - actual| cases)

In [ ]:
import sys
sys.path.insert(0, "..")

import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import shap
from sklearn.inspection import PartialDependenceDisplay

from src.models.train import (
    FEATURE_COLS,
    load_and_split,
    build_xgb_pipeline,
)
from src.models.evaluate import (
    compute_shap,
    feature_ablation,
    rolling_window_drift,
    FEATURE_GROUPS,
)
from src.models.predict import load_best_pipeline
from src.analysis.insights import home_advantage_trend, back_to_back_analysis

FEATURES_PATH = "../data/processed/features.csv"

## 1. Load Best Model + Test Set

In [ ]:
pipeline = load_best_pipeline()  # Loads best model by val Brier from MLflow

X_train, y_train, X_val, y_val, X_test, y_test = load_and_split(FEATURES_PATH)

# Full features df for insight functions
features_df = pd.read_csv(FEATURES_PATH, dtype={"GAME_ID": str})

print(f"Test set: {len(X_test):,} rows")
print(f"Pipeline steps: {list(pipeline.named_steps.keys())}")

## 2. Era Drift Analysis

Does 2003-2010 data help or hurt generalization on 2019+ games?
Train on 4-season rolling windows, evaluate on next season.

In [ ]:
df_no_bubble = features_df[features_df["is_bubble_game"] != 1].copy()

drift_df = rolling_window_drift(
    df=df_no_bubble,
    model_builder=build_xgb_pipeline,
    feature_cols=FEATURE_COLS,
    target_col="HOME_WIN",
    window_seasons=4,
    step=1,
)

drift_df.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(drift_df["val_season"], drift_df["brier"], marker="o", linewidth=2, label="Val Brier")

ax.axvspan(
    drift_df["val_season"].tolist().index("2019-20") if "2019-20" in drift_df["val_season"].values else 0,
    drift_df["val_season"].tolist().index("2019-20") if "2019-20" in drift_df["val_season"].values else 0,
    alpha=0.0
)

ax.set_xlabel("Validation Season")
ax.set_ylabel("Brier Score")
ax.set_title("Era Drift Analysis: Brier Score by Validation Season (4-season rolling window)")
ax.tick_params(axis="x", rotation=45)
ax.legend()
plt.tight_layout()
plt.savefig("../notebooks/era_drift.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nKey finding:")
early = drift_df[drift_df["val_season"] <= "2010-11"]["brier"].mean()
recent = drift_df[drift_df["val_season"] >= "2018-19"]["brier"].mean()
print(f"  Avg Brier on pre-2011 validation seasons: {early:.4f}")
print(f"  Avg Brier on post-2018 validation seasons: {recent:.4f}")
print(f"  Change: {recent - early:+.4f}")

## 3. SHAP Feature Importance (XGBoost)

In [ ]:
# Use a sample of test data for SHAP (full set is fine for TreeExplainer)
SHAP_SAMPLE = min(2000, len(X_test))
X_shap = X_test.iloc[:SHAP_SAMPLE].copy()

shap_values = compute_shap(pipeline, X_shap, model_type="xgb")

print(f"SHAP values shape: {shap_values.values.shape}")

In [ ]:
# Bar chart: mean |SHAP|
shap.plots.bar(shap_values, max_display=20, show=False)
plt.title("Mean |SHAP| - XGBoost Feature Importance")
plt.tight_layout()
plt.savefig("../notebooks/shap_bar.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Beeswarm
shap.plots.beeswarm(shap_values, max_display=20, show=False)
plt.title("SHAP Beeswarm - XGBoost (Test Set)")
plt.tight_layout()
plt.savefig("../notebooks/shap_beeswarm.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Partial Dependence Plots (Top 3 SHAP Features)

In [ ]:
import numpy as np

mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
top3_idx = np.argsort(mean_abs_shap)[::-1][:3]
top3_features = [FEATURE_COLS[i] for i in top3_idx]
print(f"Top 3 features by mean |SHAP|: {top3_features}")

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

# PDP requires the full pipeline to be passed (uses predict_proba internally)
# We need to impute first since PDP doesn't handle NaN
from sklearn.impute import SimpleImputer
import numpy as np

X_train_arr = X_train.copy()
imp = SimpleImputer(strategy="median")
X_train_imp = pd.DataFrame(imp.fit_transform(X_train_arr), columns=FEATURE_COLS)
X_test_imp  = pd.DataFrame(imp.transform(X_test),          columns=FEATURE_COLS)

# Build feature indices for the imputed array
top3_col_idx = [FEATURE_COLS.index(f) for f in top3_features]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
display = PartialDependenceDisplay.from_estimator(
    pipeline,
    X_test,
    features=top3_features,
    target=1,
    ax=axes,
    kind="average",
    grid_resolution=30,
)
for ax, feat in zip(axes, top3_features):
    ax.set_title(f"PDP: {feat}")
    ax.set_ylabel("P(home win)")

plt.suptitle("Partial Dependence Plots - Top 3 SHAP Features", y=1.02)
plt.tight_layout()
plt.savefig("../notebooks/pdp_top3.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Feature Group Ablation Study

In [ ]:
ablation_df = feature_ablation(
    pipeline_builder=build_xgb_pipeline,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    feature_groups=FEATURE_GROUPS,
)

ablation_df

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

colors = ["#d62728" if v > 0 else "#2ca02c" for v in ablation_df["brier_delta"]]
bars = ax.barh(ablation_df["group"], ablation_df["brier_delta"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Brier score delta (positive = feature group hurts performance)")
ax.set_title("Feature Group Ablation: Brier Score Impact")

for bar, val in zip(bars, ablation_df["brier_delta"]):
    ax.text(
        bar.get_width() + 0.0001,
        bar.get_y() + bar.get_height() / 2,
        f"{val:+.4f}",
        va="center",
        fontsize=9,
    )

plt.tight_layout()
plt.savefig("../notebooks/feature_ablation.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Home Advantage Trend

In [ ]:
trend_df = home_advantage_trend(features_df)
trend_df

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

# Base line
ax.plot(trend_df["season"], trend_df["home_win_rate"], marker="o", linewidth=2, color="steelblue")

# Annotate COVID seasons
for _, row in trend_df[trend_df["is_no_fans_season"]].iterrows():
    ax.annotate(
        "No fans",
        xy=(row["season"], row["home_win_rate"]),
        xytext=(0, 15),
        textcoords="offset points",
        ha="center",
        fontsize=8,
        color="red",
        arrowprops=dict(arrowstyle="->", color="red"),
    )

ax.axhline(0.5, color="gray", linestyle="--", linewidth=1, label="50% baseline")
ax.set_xlabel("Season")
ax.set_ylabel("Home Win Rate")
ax.set_title("Home Advantage Trend (excluding bubble games)")
ax.tick_params(axis="x", rotation=45)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
ax.legend()
plt.tight_layout()
plt.savefig("../notebooks/home_advantage_trend.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nPre-2010 avg home win rate: {trend_df[trend_df['season'] < '2010-11']['home_win_rate'].mean():.3f}")
print(f"Post-2018 avg home win rate: {trend_df[trend_df['season'] >= '2018-19']['home_win_rate'].mean():.3f}")

## 7. B2B Penalty Analysis

In [ ]:
b2b_df = back_to_back_analysis(features_df, pipeline)
b2b_df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

bars = ax.bar(
    b2b_df["scenario"],
    b2b_df["mean_predicted_prob"],
    yerr=b2b_df["std_predicted_prob"] / np.sqrt(b2b_df["n_games"]),
    capsize=5,
    color=["steelblue", "tomato", "orange", "gray"],
    alpha=0.85,
)

ax.axhline(0.5, color="black", linestyle="--", linewidth=0.8, label="50%")
ax.set_ylabel("Mean P(home win)")
ax.set_title("Predicted Home Win Probability by B2B Scenario")
ax.set_ylim([0.4, 0.7])
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=1))
ax.tick_params(axis="x", rotation=20)

for bar, row in zip(bars, b2b_df.itertuples()):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.003,
        f"n={row.n_games:,}",
        ha="center", va="bottom", fontsize=8,
    )

ax.legend()
plt.tight_layout()
plt.savefig("../notebooks/b2b_penalty.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. is_no_fans_season SHAP Effect

In [ ]:
# Pull SHAP values for is_no_fans_season
no_fans_idx = FEATURE_COLS.index("is_no_fans_season")
no_fans_shap = shap_values.values[:, no_fans_idx]
no_fans_feat = X_shap["is_no_fans_season"].values

fig, ax = plt.subplots(figsize=(6, 4))
for val, label in [(0, "Normal season"), (1, "No fans (2020-21)")]:
    mask = no_fans_feat == val
    if mask.sum() > 0:
        ax.hist(no_fans_shap[mask], bins=30, alpha=0.7, label=label)

ax.axvline(0, color="black", linewidth=1)
ax.set_xlabel("SHAP value for is_no_fans_season")
ax.set_ylabel("Count")
ax.set_title("Model's Learned Effect: is_no_fans_season")
ax.legend()
plt.tight_layout()
plt.savefig("../notebooks/no_fans_shap.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Mean SHAP (normal seasons): {no_fans_shap[no_fans_feat == 0].mean():.4f}")
print(f"Mean SHAP (no fans season): {no_fans_shap[no_fans_feat == 1].mean():.4f}")

## 9. Error Analysis

Identify games where |predicted_prob - actual| > 0.4. Are they concentrated in specific seasons?

In [ ]:
test_df = features_df[
    (features_df["is_bubble_game"] != 1) &
    features_df["SEASON"].isin(["2020-21", "2021-22"])
].copy()

test_probs = pipeline.predict_proba(test_df[[c for c in FEATURE_COLS if c in test_df.columns]])[:, 1]
test_df["predicted_prob"] = test_probs
test_df["abs_error"] = (test_df["predicted_prob"] - test_df["HOME_WIN"]).abs()

high_error = test_df[test_df["abs_error"] > 0.4].copy()
print(f"Games with |error| > 0.4: {len(high_error)} ({len(high_error)/len(test_df)*100:.1f}%)")

print("\nBy season:")
print(high_error.groupby("SEASON")["abs_error"].agg(["count", "mean"]).round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Error distribution
axes[0].hist(test_df["abs_error"], bins=40, edgecolor="none", color="steelblue", alpha=0.8)
axes[0].axvline(0.4, color="red", linestyle="--", label="0.4 threshold")
axes[0].set_xlabel("|predicted_prob - actual|")
axes[0].set_ylabel("Count")
axes[0].set_title("Absolute Error Distribution (Test Set)")
axes[0].legend()

# Error by season
season_err = test_df.groupby("SEASON")["abs_error"].mean()
axes[1].bar(season_err.index, season_err.values, color="tomato", alpha=0.8)
axes[1].set_xlabel("Season")
axes[1].set_ylabel("Mean |error|")
axes[1].set_title("Mean Absolute Error by Season (Test Set)")

plt.tight_layout()
plt.savefig("../notebooks/error_analysis.png", dpi=150, bbox_inches="tight")
plt.show()